In [1]:
import h5py
import pysam

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix,csc_matrix

import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
import os
import anndata

In [2]:
#loading pmat 

In [3]:
ad=sc.read_h5ad('./Humanbrain.sample.h5ad')
ad

AnnData object with n_obs × n_vars = 5004 × 228790

In [4]:
peaks=pd.read_csv('peaks.csv')
peaks

,chr,start,end,feature
0,chr1,179780,181780,chr1-179780-181780
1,chr1,190494,192494,chr1-190494-192494
2,chr1,633024,635024,chr1-633024-635024
3,chr1,777725,779725,chr1-777725-779725
4,chr1,778185,780185,chr1-778185-780185
...,...,...,...,...
228785,chrY,7342419,7344419,chrY-7342419-7344419
228786,chrY,13702403,13704403,chrY-13702403-13704403
228787,chrY,13868376,13870376,chrY-13868376-13870376
228788,chrY,15455627,15457627,chrY-15455627-15457627


In [5]:
ad.var_names = peaks['feature'].values

In [6]:
ad.var['features'] = ad.var_names
ad.var

,features
chr1-179780-181780,chr1-179780-181780
chr1-190494-192494,chr1-190494-192494
chr1-633024-635024,chr1-633024-635024
chr1-777725-779725,chr1-777725-779725
chr1-778185-780185,chr1-778185-780185
...,...
chrY-7342419-7344419,chrY-7342419-7344419
chrY-13702403-13704403,chrY-13702403-13704403
chrY-13868376-13870376,chrY-13868376-13870376
chrY-15455627-15457627,chrY-15455627-15457627


1.shuffle

In [7]:
## shuffle ad.var
import random

def shuffleData(ad, seed=666):
    random.seed(seed)
    index = [i for i in range(len(ad.var))]
    random.shuffle(index)
    ad_shuffled = ad[:,index]
    return ad_shuffled

In [8]:
ad = shuffleData(ad)
ad

View of AnnData object with n_obs × n_vars = 5004 × 228790
    var: 'features'

In [9]:
ad.var

,features
chr12-131610284-131612284,chr12-131610284-131612284
chr8-107982545-107984545,chr8-107982545-107984545
chrX-37587948-37589948,chrX-37587948-37589948
chr7-99437877-99439877,chr7-99437877-99439877
chr4-82191015-82193015,chr4-82191015-82193015
...,...
chr3-197553889-197555889,chr3-197553889-197555889
chr17-1030922-1032922,chr17-1030922-1032922
chr22-17577316-17579316,chr22-17577316-17579316
chr2-145116073-145118073,chr2-145116073-145118073


In [10]:
feature = ad.var['features'].str.split(pat='-', n=-1, expand=True).rename(columns={0:'chr', 1:'start', 2:'end'}) 
feature['features'] = feature.chr+'-'+feature.start+'-'+feature.end
feature

,chr,start,end,features
chr12-131610284-131612284,chr12,131610284,131612284,chr12-131610284-131612284
chr8-107982545-107984545,chr8,107982545,107984545,chr8-107982545-107984545
chrX-37587948-37589948,chrX,37587948,37589948,chrX-37587948-37589948
chr7-99437877-99439877,chr7,99437877,99439877,chr7-99437877-99439877
chr4-82191015-82193015,chr4,82191015,82193015,chr4-82191015-82193015
...,...,...,...,...
chr3-197553889-197555889,chr3,197553889,197555889,chr3-197553889-197555889
chr17-1030922-1032922,chr17,1030922,1032922,chr17-1030922-1032922
chr22-17577316-17579316,chr22,17577316,17579316,chr22-17577316-17579316
chr2-145116073-145118073,chr2,145116073,145118073,chr2-145116073-145118073


In [11]:
ad.var['chr'] = feature['chr'].values
ad.var['start'] = feature['start'].values
ad.var['end'] = feature['end'].values

/tmp/ipykernel_817510/2563709048.py:1: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  ad.var['chr'] = feature['chr'].values


In [12]:
ad.var

,features,chr,start,end
chr12-131610284-131612284,chr12-131610284-131612284,chr12,131610284,131612284
chr8-107982545-107984545,chr8-107982545-107984545,chr8,107982545,107984545
chrX-37587948-37589948,chrX-37587948-37589948,chrX,37587948,37589948
chr7-99437877-99439877,chr7-99437877-99439877,chr7,99437877,99439877
chr4-82191015-82193015,chr4-82191015-82193015,chr4,82191015,82193015
...,...,...,...,...
chr3-197553889-197555889,chr3-197553889-197555889,chr3,197553889,197555889
chr17-1030922-1032922,chr17-1030922-1032922,chr17,1030922,1032922
chr22-17577316-17579316,chr22-17577316-17579316,chr22,17577316,17579316
chr2-145116073-145118073,chr2-145116073-145118073,chr2,145116073,145118073


In [13]:
peaks = ad.var.iloc[:,1:4]
peaks

,chr,start,end
chr12-131610284-131612284,chr12,131610284,131612284
chr8-107982545-107984545,chr8,107982545,107984545
chrX-37587948-37589948,chrX,37587948,37589948
chr7-99437877-99439877,chr7,99437877,99439877
chr4-82191015-82193015,chr4,82191015,82193015
...,...,...,...
chr3-197553889-197555889,chr3,197553889,197555889
chr17-1030922-1032922,chr17,1030922,1032922
chr22-17577316-17579316,chr22,17577316,17579316
chr2-145116073-145118073,chr2,145116073,145118073


In [14]:
#input

In [15]:
onehot_nuc = {'A':[1,0,0,0],
            'C':[0,1,0,0],
            'G':[0,0,1,0],
            'T':[0,0,0,1],
            'N':[0,0,0,0]}
            
def _onehot_seq(seq):
    return np.array([onehot_nuc[nuc] for nuc in str(seq).upper()])

In [16]:
# genome
genome = pysam.Fastafile('/media/ggj/FYT/CH/CH-cross/cluster/compare/fasta/hg38/hg38.fa')
genome

In [17]:
from tqdm import tqdm

In [18]:
seq_onehot = []

for peak in tqdm(peaks.values):
    seqnames, start, end = peak[:3]
    #chrom = str(seqnames.replace("chr",""))
    start, end = int(start), int(end)
    chrom = seqnames
    # catch overflowed error
    chrom_size = genome.get_reference_length(chrom)
    if end > chrom_size:
        print(peak[-1])
        pad = 'N' * (end - chrom_size) # pad N
        end = chrom_size
    # fetch sequence 
    seq = genome.fetch(reference=chrom, start=start, end=end)
    # pad N
    if start + 2000 > chrom_size:
        seq += pad
    # onehot    
    seq = _onehot_seq(seq)
    seq_onehot.append(seq)

seq_onehot = np.array(seq_onehot, dtype=np.bool)

seq_onehot.shape

100%|██████████████████████████████████| 228790/228790 [04:53<00:00, 779.71it/s]
/tmp/ipykernel_817510/2381727549.py:23: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  seq_onehot = np.array(seq_onehot, dtype=np.bool)


(228790, 2000, 4)

In [19]:
seq_onehot.shape

(228790, 2000, 4)

# 4.Split train/valid/test data

In [20]:
from sklearn.model_selection import train_test_split

In [21]:
idx = peaks.index.values
mask = peaks.chr.astype(str).isin(['chr8','chr7'])

test_idx = idx[mask]

In [22]:
mask = peaks.chr.astype(str).isin(['chr9'])
val_idx = idx[mask]

mask = peaks.chr.astype(str).isin(['chr9','chr8','chr7'])
train_idx = idx[~mask]
train_idx.shape, val_idx.shape, test_idx.shape

((192044,), (10004,), (26742,))

In [23]:
peaks['train_test_split'] = "train"
peaks.loc[train_idx, 'train_test_split'] = "train"
peaks.loc[val_idx, 'train_test_split'] = "val"
peaks.loc[test_idx, 'train_test_split'] = "test"

peaks.head()

,chr,start,end,train_test_split
chr12-131610284-131612284,chr12,131610284,131612284,train
chr8-107982545-107984545,chr8,107982545,107984545,test
chrX-37587948-37589948,chrX,37587948,37589948,train
chr7-99437877-99439877,chr7,99437877,99439877,test
chr4-82191015-82193015,chr4,82191015,82193015,train


In [24]:
# create coordinate sparse matrix
pmat_co = coo_matrix(ad.X.T)

In [25]:
pmat_co.shape

(228790, 5004)

# 5.Write

In [26]:
## 5.Write
compress_args = {'compression': 'gzip', 'compression_opts': 1}

h5file = h5py.File('Humanbrain.sample.h5', 'a')
h5file.create_dataset("pmat/X", data=seq_onehot, dtype=bool, **compress_args)
h5file.create_dataset("pmat/peak", data=peaks.values.astype(np.bytes_), **compress_args)

#h5file.create_dataset("pmat/pmat_sc/y", data=y, dtype=np.float32, **compress_args)
h5file.create_dataset("pmat/pmat_sc/i", data=pmat_co.row, dtype=np.int32, **compress_args)
h5file.create_dataset("pmat/pmat_sc/j", data=pmat_co.col, dtype=np.int32, **compress_args)
h5file.create_dataset("pmat/pmat_sc/x", data=pmat_co.data, dtype=np.int32, **compress_args)
h5file.create_dataset("pmat/pmat_sc/dim", data=pmat_co.shape, dtype=np.int32, **compress_args)

# h5file.create_dataset("cellanno", data=anno.values.astype(np.bytes_), **compress_args)

h5file.close()